In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
paths = get_paths(ROOT)

paths

ProjectPaths(root=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526'), src=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/src'), notebooks=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/notebooks'), data_processed=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/processed'), data_raw=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/raw'), checkpoints=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/checkpoints'))

In [2]:
from kaggle_download import download_kaggle_dataset_once

download_kaggle_dataset_once(
    dataset_slug= "rmisra/imdb-spoiler-dataset",
    raw_dir=paths.data_raw,
    project_root = paths.root,
    force=False
)

list(paths.data_raw.iterdir())[:10]

[WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/raw/IMDB_movie_details.json'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/raw/IMDB_reviews.json')]

In [3]:
from imdb_spoiler_io import load_raw_imdb_spoiler_json, prepare_reviews_dataframe, save_processed_reviews

df_raw = load_raw_imdb_spoiler_json(paths.data_raw)
df, schema = prepare_reviews_dataframe(df_raw)

print(schema)
df.head(), df["label"].value_counts(normalize=True)


ImdbSpoilerSchema(text_col='text', label_col='label', movie_id_col='movie_id')


(      review_id   movie_id                                               text  \
 1572          0  tt0111161  In its Oscar year, Shawshank Redemption (writt...   
 1573          1  tt0111161  The Shawshank Redemption is without a doubt on...   
 1574          2  tt0111161  I believe that this film is the best story eve...   
 1575          3  tt0111161  **Yes, there are SPOILERS here**This film has ...   
 1576          4  tt0111161  At the heart of this extraordinary movie is a ...   
 
       label  
 1572      1  
 1573      1  
 1574      1  
 1575      1  
 1576      1  ,
 label
 0    0.737026
 1    0.262974
 Name: proportion, dtype: float64)

In [4]:
out = save_processed_reviews(df, paths.data_processed, save_csv=False)  # csv opzionale
out


WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/processed/reviews.parquet')

In [5]:
from splitters import SplitConfig, split_by_movie_id, save_splits

cfg = SplitConfig(train_size=0.8, val_size=0.1, test_size=0.1, seed=42, max_tries=500)
train_df, val_df, test_df = split_by_movie_id(df, cfg)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("label mean train/val/test:",
      train_df["label"].mean(), val_df["label"].mean(), test_df["label"].mean())

print("unique movies train/val/test:",
      train_df["movie_id"].nunique(), val_df["movie_id"].nunique(), test_df["movie_id"].nunique())

save_splits(train_df, val_df, test_df, paths.data_processed)
list(paths.data_processed.iterdir())


train: 467362 val: 52251 test: 54300
label mean train/val/test: 0.2629717435307107 0.2629806128112381 0.26298342541436465
unique movies train/val/test: 1258 157 157


[WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/processed/reviews.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/processed/test.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/processed/train.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/processed/val.parquet')]